# SQL Practice Notebook (Databricks)

In this notebook, we will cover:

- Creating tables
- Querying data
- Filtering, sorting
- Aggregations
- Joins
- Window functions
- CTEs and subqueries
- Writing data

All examples follow real-world data engineering scenarios.

## Step 1: Create Raw Table (Bronze Layer)

We simulate raw ingested data.

In [0]:
CREATE OR REPLACE TABLE sales_raw (
    order_id INT,
    customer_id INT,
    product STRING,
    category STRING,
    amount DOUBLE,
    order_date STRING,
    country STRING
);

In [0]:
INSERT INTO sales_raw VALUES
(1, 101, 'Laptop', 'Electronics', 1200, '2024-01-01', 'UK'),
(2, 102, 'Phone', 'Electronics', 800, '2024-01-02', 'UK'),
(3, 101, 'Tablet', 'Electronics', 600, '2024-01-03', 'India'),
(4, 103, 'Shoes', 'Fashion', 150, '2024-01-04', 'USA'),
(5, 104, 'Watch', 'Fashion', 200, '2024-01-05', 'UK'),
(6, 102, 'Laptop', 'Electronics', 1100, '2024-01-06', 'India');

## Step 2: Basic SELECT Queries

In [0]:
SELECT * FROM sales_raw;

In [0]:
SELECT product, amount FROM sales_raw;

## Step 3: Filtering Data

In [0]:
SELECT *
FROM sales_raw
WHERE amount > 500;

In [0]:
SELECT *
FROM sales_raw
WHERE country = 'UK' AND amount > 500;

## Step 4: Sorting Results

In [0]:
SELECT *
FROM sales_raw
ORDER BY amount DESC;

## Step 5: Aggregations

In [0]:
SELECT
    category,
    SUM(amount) AS total_sales,
    AVG(amount) AS avg_sales,
    COUNT(*) AS total_orders
FROM sales_raw
GROUP BY category;

## Step 6: HAVING Clause

In [0]:
SELECT category, SUM(amount) AS total
FROM sales_raw
GROUP BY category
HAVING total > 1000;

## Step 7: Create Customer Table (Dimension)

In [0]:
CREATE OR REPLACE TABLE customers (
    customer_id INT,
    customer_name STRING,
    segment STRING
);

In [0]:
INSERT INTO customers VALUES
(101, 'Alice', 'Premium'),
(102, 'Bob', 'Standard'),
(103, 'Charlie', 'Premium'),
(104, 'David', 'Standard');

## Step 8: Joins

In [0]:
SELECT
    s.order_id,
    c.customer_name,
    s.amount
FROM sales_raw s
INNER JOIN customers c
ON s.customer_id = c.customer_id;

In [0]:
-- Left Join
SELECT *
FROM sales_raw s
LEFT JOIN customers c
ON s.customer_id = c.customer_id;

## Step 9: Conditional Logic

In [0]:
SELECT *,
    CASE
        WHEN amount > 1000 THEN 'High'
        WHEN amount > 500 THEN 'Medium'
        ELSE 'Low'
    END AS sales_category
FROM sales_raw;

## Step 10: Date Functions

In [0]:
SELECT *,
    TO_DATE(order_date) AS order_dt,
    YEAR(TO_DATE(order_date)) AS year,
    MONTH(TO_DATE(order_date)) AS month
FROM sales_raw;

## Step 11: Window Functions

In [0]:
SELECT *,
    ROW_NUMBER() OVER (PARTITION BY category ORDER BY amount DESC) AS rank
FROM sales_raw;

In [0]:
SELECT *,
    SUM(amount) OVER (PARTITION BY category) AS category_total
FROM sales_raw;

## Step 12: CTE

In [0]:
WITH sales_summary AS (
    SELECT category, SUM(amount) AS total_sales
    FROM sales_raw
    GROUP BY category
)
SELECT *
FROM sales_summary
WHERE total_sales > 1000;

## Step 13: Subqueries

In [0]:
SELECT *
FROM sales_raw
WHERE amount > (
    SELECT AVG(amount) FROM sales_raw
);

## Step 14: Create Cleaned Table (Silver Layer)

In [0]:
CREATE OR REPLACE TABLE sales_clean AS
SELECT
    order_id,
    customer_id,
    product,
    category,
    amount,
    TO_DATE(order_date) AS order_date,
    country
FROM sales_raw;

## Step 15: Create Aggregated Table (Gold Layer)


In [0]:
CREATE OR REPLACE TABLE sales_gold AS
SELECT
    category,
    country,
    SUM(amount) AS total_sales,
    COUNT(*) AS total_orders
FROM sales_clean
GROUP BY category, country;

## Step 16: Query Gold Table

In [0]:
SELECT * FROM sales_gold;

## Step 17: Create View

In [0]:
CREATE OR REPLACE VIEW sales_view AS
SELECT * FROM sales_gold;

## Step 18: Optimize Table (Delta Lake)

In [0]:
OPTIMIZE sales_clean;

In [0]:
-- Z-Ordering
OPTIMIZE sales_clean ZORDER BY (customer_id);

## Key Learnings

- SQL is the backbone of data engineering
- Window functions are heavily used in interviews
- Always think in layers:
    - Bronze → Raw
    - Silver → Cleaned
    - Gold → Aggregated
- Use CTEs for readability
- Optimize tables for performance in Databricks